In [1]:
import torch
import torchvision
import torch.nn as nn
import cv2
import random
import json
from typing import Literal
from pydantic import BaseModel
from pathlib import Path
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from  torchvision import models 
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
def train_val_test_split(
    csv_path: str | Path,
    train_matches: list[str],
    val_matches: list[str],
    test_matches: list[str],
    match_col: str = "match",
    img_name_col: str = "img_name",
) -> dict[str, list[str]]:
    """
    Dzieli dane na train/val/test na podstawie ręcznie podanych meczów.
    Przyjmuje ścieżkę do CSV z shot_labels (kolumny: img_name, label, match).
    Zwraca słownik: {"train": [nazwy zdjęć], "val": [...], "test": [...]}.
    """
    df = pd.read_csv(csv_path)
    
    if match_col not in df.columns:
        raise ValueError(f"Kolumna '{match_col}' nie istnieje w CSV. Dostępne: {list(df.columns)}")
    if img_name_col not in df.columns:
        raise ValueError(f"Kolumna '{img_name_col}' nie istnieje w CSV.")

    def img_names_for_matches(matches: list[str]) -> list[str]:
        mask = df[match_col].isin(matches)
        return df.loc[mask, img_name_col].astype(str).tolist()

    return {
        "train": img_names_for_matches(train_matches),
        "val": img_names_for_matches(val_matches),
        "test": img_names_for_matches(test_matches),
    }

In [3]:
df_labels = pd.read_csv('shot_labels.csv')

In [4]:
df_labels[['match']].value_counts()

match   
m_pic       608
skip_m07    273
skip_m05    120
skip_m03    113
skip_m02    113
m02          99
skip_m09     60
m05          57
skip_m10     52
m04          50
m06          49
skip_m01     46
skip_m08     43
m01          40
skip_m04     37
m03          36
skip_m06     14
Name: count, dtype: int64

In [5]:
train_matches = ["m_pic", 'm01', 'm02', 'm03', "skip_m01", "skip_m02", 'skip_m03', "skip_m04", "skip_m05", "skip_m06", "skip_m07"] 
val_matches = ['m04', 'm05', 'm06', 'skip_m08', 'skip_m09', 'skip_m10'] 
test_matches = [ ]

split_spec = train_val_test_split(
    "shot_labels.csv",
    train_matches=train_matches,
    val_matches=val_matches,
    test_matches=test_matches,
)


with open("split_spec.json", "w", encoding="utf-8") as f:
    json.dump(split_spec, f, indent=4, ensure_ascii=False)

full_df = pd.read_csv("shot_labels.csv")
train_df = full_df[full_df["img_name"].isin(split_spec["train"])].reset_index(drop=True)
val_df = full_df[full_df["img_name"].isin(split_spec["val"])].reset_index(drop=True)
test_df = full_df[full_df["img_name"].isin(split_spec["test"])].reset_index(drop=True)

In [6]:
print(train_df['label'].value_counts(normalize=True))
print(val_df['label'].value_counts(normalize=True))
print(test_df['label'].value_counts(normalize=True))

label
1    0.522348
0    0.477652
Name: proportion, dtype: float64
label
1    0.501608
0    0.498392
Name: proportion, dtype: float64
Series([], Name: proportion, dtype: float64)


In [7]:
print(len(train_df))
print(len(val_df))

1499
311


In [8]:
def image_to_tensor(img: np.ndarray, resize: tuple[int, int] | None = None, with_batch_size: bool = True) -> torch.Tensor:
    if resize is not None:
        img = cv2.resize(img, resize)

    if img.ndim == 2:
        img = img[:, :, np.newaxis]

    img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
    return img_tensor.unsqueeze(0) if with_batch_size else img_tensor


def tensor_to_image(img_tensor: torch.Tensor) -> np.ndarray:
    if img_tensor.shape[0] > 1:
        raise ValueError("batch size must be equals to 1")
    
    img_arr = img_tensor.detach().numpy()
    return img_arr.squeeze()


def prepare_img_for_resnet(img: np.ndarray):
    img = cv2.resize(img, (224, 224)).astype(np.float32)
    img /= 255
    img_tensor = torch.tensor(img).permute(2, 0, 1).float()# .unsqueeze(0)
    return torchvision.transforms.functional.normalize(img_tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

In [9]:
class SnookerShotDataset(Dataset):
    def __init__(self, data_labels_path: str | Path, split_spec_path: str | Path, pics_root: str | Path = "../pics", mode: Literal['train', 'val', 'test'] = 'train'):
        self.pics_root = Path(pics_root)

        data = pd.read_csv(data_labels_path)
        split_spec = json.load(open(split_spec_path))
        self.data = data[data["img_name"].isin(split_spec[mode])].sample(frac=1).reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.data)

    def _img_path(self, img_name: str) -> Path:
        subdir = "skip" if "skip_" in img_name else "."
        return self.pics_root / subdir / f"{img_name}.png"

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.data.iloc[idx]
        img_path = self._img_path(row["img_name"])

        img = cv2.imread(str(img_path)).astype(np.float32)
        img_tensor = prepare_img_for_resnet(img)
        label = torch.tensor(row["label"], dtype=torch.float32)

        return img_tensor, label

In [10]:
class History(BaseModel):
    epoch_losses: list[float] = []
    epoch_accuracy: list[float] = []
    step_losses: list[float] = []
    best_loss: float = np.inf
    best_accuracy: float = 0
    _running_loss: float
    _correct_pred: float

    def on_epoch_start(self) -> None:
        self._running_loss = .0
        self._correct_pred = .0

    def on_epoch_end(self, dataset: Dataset, focus: Literal['loss', 'accuracy']) -> bool:
        """
        Czy dana epoka polepszyla model wzgledem treningu na podstawie metryki przekazywanej w argumencie focus
        """
        current_loss = self._running_loss / len(dataset)
        current_accuracy = self._correct_pred / len(dataset)
        self.epoch_losses.append(current_loss)
        self.epoch_accuracy.append(current_accuracy)

        if is_better_loss := current_loss < self.best_loss:
            self.best_loss = current_loss
  
        if is_better_accuracy := current_accuracy > self.best_accuracy:
            self.best_accuracy = current_accuracy

        return  {'loss': is_better_loss, 'accuracy': is_better_accuracy}[focus]

        
    def on_step_end(self, loss: float, y_hat: torch.Tensor, y_gt: torch.Tensor) -> None:
        y_pred = (F.sigmoid(y_hat).reshape(-1) >= 0.5).long()
        self._correct_pred += (y_pred == y_gt).sum().item()
        self._running_loss += loss
        self.step_losses.append(loss)

        

    def get_latest(self, mode: Literal['train', 'val']) -> str:
        return f"{mode} loss= {self.epoch_losses[-1]:.4f} {mode} accuracy= {self.epoch_accuracy[-1]:.4f}"

In [11]:
train_dataset = SnookerShotDataset("shot_labels.csv", "split_spec.json", mode="train")
val_dataset = SnookerShotDataset("shot_labels.csv", "split_spec.json", mode="val")
# test_dataset = SnookerShotDataset("shot_labels.csv", "split_spec.json", mode="test")

In [12]:
train_dataloader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=128)
# test_dataloader = DataLoader(test_dataset, batch_size=32)

In [13]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

In [14]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [15]:
model.fc.in_features # liczba cech do glowy

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 1),
)

In [16]:
# model

In [17]:
for name, param in model.named_parameters():
    if not name.startswith(('fc', 
                            'layer4.1.conv2.weight', 'layer4.1.bn2.weight', 'layer4.1.bn2.bias', 
                             'layer4.1.conv1.weight', 'layer4.1.bn1.weight', 'layer4.1.bn1.bias', 
                            
                            'layer4.0.conv1.weight', 'layer4.0.bn1.weight', 'layer4.0.bn1.bias',
                            'layer4.0.conv2.weight', 'layer4.0.bn2.weight', 'layer4.0.bn2.bias'
                            
                            )): 
        param.requires_grad = False

In [ ]:
torch.manual_seed(123)
np.random.seed(123)
random.seed(123)


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
epochs_num = 10
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=4)
focus = "loss"
model = model.to(device)

train_hist = History()
val_hist = History()
learning_rates = []

for ep in range(epochs_num):
    model.train()

    train_hist.on_epoch_start()

    for batch_x, batch_y in train_dataloader:
        optimizer.zero_grad()
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)  
        y_hat = model(batch_x)


        loss = criterion(y_hat, batch_y.reshape(-1, 1))
        loss.backward()
        optimizer.step()
        train_hist.on_step_end(loss.item(), y_hat, batch_y)

    train_hist.on_epoch_end(train_dataset, focus)
    model.eval()
    val_hist.on_epoch_start()
    with torch.no_grad():

        for batch_x, batch_y in val_dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            y_hat = model(batch_x)

            loss = criterion(y_hat, batch_y.reshape(-1, 1))
            val_hist.on_step_end(loss.item(), y_hat, batch_y)


    save_best_model = val_hist.on_epoch_end(val_dataset, focus)
    scheduler.step(val_hist.epoch_losses[-1])
    learning_rates.append(optimizer.param_groups[0]['lr'])
    print(f"Epoka {ep+1}/{epochs_num}: {train_hist.get_latest(mode='train')}, {val_hist.get_latest('val')}")

    checkpoint_data = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": ep + 1}
    torch.save(checkpoint_data, '../models/shot-classifier-last.pt')
    if save_best_model:
        torch.save(checkpoint_data, '../models/shot-classifier-best.pt')


Epoka 1/10: train loss= 0.0009 train accuracy= 0.5103, val loss= 0.0061 val accuracy= 0.5949
Epoka 2/10: train loss= 0.0007 train accuracy= 0.8419, val loss= 0.0051 val accuracy= 0.9518
Epoka 3/10: train loss= 0.0006 train accuracy= 0.9506, val loss= 0.0042 val accuracy= 0.9678
Epoka 4/10: train loss= 0.0005 train accuracy= 0.9833, val loss= 0.0034 val accuracy= 0.9711
Epoka 5/10: train loss= 0.0004 train accuracy= 0.9913, val loss= 0.0028 val accuracy= 0.9775
Epoka 6/10: train loss= 0.0003 train accuracy= 0.9933, val loss= 0.0024 val accuracy= 0.9775
Epoka 7/10: train loss= 0.0003 train accuracy= 0.9987, val loss= 0.0021 val accuracy= 0.9775
Epoka 8/10: train loss= 0.0002 train accuracy= 0.9987, val loss= 0.0018 val accuracy= 0.9807
Epoka 9/10: train loss= 0.0002 train accuracy= 0.9993, val loss= 0.0016 val accuracy= 0.9807
Epoka 10/10: train loss= 0.0002 train accuracy= 0.9993, val loss= 0.0015 val accuracy= 0.9807


In [40]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


img = cv2.imread('/home/polymorvic/development/deep-peep-snooker/skip_test.png').astype(np.float32)  # skip_test
img = prepare_img_for_resnet(img)
img = img.unsqueeze(0)
img = img.to(device)

with torch.no_grad():
    y_hat = model(img)
    probs = F.sigmoid(y_hat)


In [41]:
probs

tensor([[0.0258]], device='cuda:0')

device(type='cuda')

In [ ]:
# load_model = models.resnet18(weights=None)


# load_model.fc = nn.Sequential(
#     nn.Linear(load_model.fc.in_features, 256),
#     nn.ReLU(),
#     nn.Dropout(0.4),
#     nn.Linear(256, 64),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(64, 1),
# )


# load_model.state_dict(torch.load('/home/polymorvic/development/deep-peep-snooker/models/v5/shot-classifier-best.pt'))

/tmp/ipykernel_10898/4220285649.py:15: FutureWarning: Positional args are being deprecated, use kwargs instead. Refer to https://pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.state_dict for details.
  load_model.state_dict(torch.load('/home/polymorvic/development/deep-peep-snooker/models/v5/shot-classifier-best.pt'))


{'model': OrderedDict([('conv1.weight',
               tensor([[[[-1.0419e-02, -6.1356e-03, -1.8098e-03,  ...,  5.6615e-02,
                           1.7083e-02, -1.2694e-02],
                         [ 1.1083e-02,  9.5276e-03, -1.0993e-01,  ..., -2.7124e-01,
                          -1.2907e-01,  3.7424e-03],
                         [-6.9434e-03,  5.9089e-02,  2.9548e-01,  ...,  5.1972e-01,
                           2.5632e-01,  6.3573e-02],
                         ...,
                         [-2.7535e-02,  1.6045e-02,  7.2595e-02,  ..., -3.3285e-01,
                          -4.2058e-01, -2.5781e-01],
                         [ 3.0613e-02,  4.0960e-02,  6.2850e-02,  ...,  4.1384e-01,
                           3.9359e-01,  1.6606e-01],
                         [-1.3736e-02, -3.6746e-03, -2.4084e-02,  ..., -1.5070e-01,
                          -8.2230e-02, -5.7828e-03]],
               
                        [[-1.1397e-02, -2.6619e-02, -3.4641e-02,  ...,  3.2521e-02,
       

In [4]:
load_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


img = cv2.imread('/home/polymorvic/development/deep-peep-snooker/skip_test.png').astype(np.float32)
img = prepare_img_for_resnet(img)
img = img.unsqueeze(0)

with torch.no_grad():
    y_hat = load_model(img)
    probs = F.sigmoid(y_hat)

NameError: name 'prepare_img_for_resnet' is not defined

In [6]:
probs

tensor([[0.5538]])

In [29]:
with open("split_spec.json", encoding="utf-8") as f:
    split_spec = json.load(f)

In [30]:
import random

In [31]:
val_pics = []
for _ in range(16):

    while True:
        pic = random.choice(split_spec['val'])
        if pic not in val_pics:
            val_pics.append(pic)
            break

In [32]:
df = pd.read_csv('shot_labels.csv')

In [33]:
val_pics

['05_0016',
 'skip_m08_0034',
 'skip_m10_0019',
 '05_0011',
 '06_0038',
 'skip_m09_0012',
 'skip_m10_0005',
 '04_0023',
 'skip_m09_0058',
 'skip_m10_0010',
 'skip_m10_0021',
 'skip_m08_0014',
 '05_0043',
 'skip_m10_0045',
 '05_0057',
 'skip_m10_0027']

In [34]:
val_df = df[df['img_name'].isin(val_pics)]

In [14]:
val_df

,img_name,label,match
369,06_0020,1,m06
412,05_0027,1,m05
461,skip_m09_0009,0,skip_m09
620,skip_m09_0008,0,skip_m09
644,06_0003,1,m06
670,skip_m09_0011,0,skip_m09
718,04_0049,1,m04
982,skip_m10_0021,0,skip_m10
1033,skip_m10_0016,0,skip_m10
1040,skip_m09_0023,0,skip_m09


In [35]:
# load_model = models.resnet18()


# load_model.fc = nn.Sequential(
#     nn.Linear(load_model.fc.in_features, 256),
#     nn.ReLU(),
#     nn.Dropout(0.4),
#     nn.Linear(256, 64),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(64, 1),
# )


# load_model.state_dict(torch.load('/home/polymorvic/development/deep-peep-snooker/models/v5/shot-classifier-best.pt', weights_only=True))

# load_model.eval()
# device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
images = []
labels = []
preds = []
for _, row in val_df.iterrows():
    img_name = row['img_name']
    dir = 'pics/' if 'skip_' not in img_name else 'pics/skip/'
    image = cv2.imread(f'../{dir}{img_name}.png')
    images.append(image)

    labels.append(row['label'])


    image = prepare_img_for_resnet(image)
    image = image.unsqueeze(0)
    image = image.to(device)

    with torch.no_grad():
        y_hat = model(image)
        # print(y_hat)
        prob = F.sigmoid(y_hat)
        print(prob)

    preds.append(prob)

    

tensor([[0.0492]], device='cuda:0')
tensor([[0.7225]], device='cuda:0')
tensor([[0.0351]], device='cuda:0')
tensor([[0.0368]], device='cuda:0')
tensor([[0.0549]], device='cuda:0')
tensor([[0.8679]], device='cuda:0')
tensor([[0.0515]], device='cuda:0')
tensor([[0.7127]], device='cuda:0')
tensor([[0.0228]], device='cuda:0')
tensor([[0.0433]], device='cuda:0')
tensor([[0.8765]], device='cuda:0')
tensor([[0.0312]], device='cuda:0')
tensor([[0.8623]], device='cuda:0')
tensor([[0.8867]], device='cuda:0')
tensor([[0.0199]], device='cuda:0')
tensor([[0.0427]], device='cuda:0')


In [ ]:
val_df

,img_name,label,match
146,skip_m10_0005,0,skip_m10
188,05_0016,1,m05
265,skip_m10_0019,0,skip_m10
348,skip_m09_0012,0,skip_m09
404,skip_m10_0045,0,skip_m10
482,05_0043,1,m05
699,skip_m10_0027,0,skip_m10
740,06_0038,1,m06
749,skip_m08_0034,0,skip_m08
982,skip_m10_0021,0,skip_m10
